In [0]:
import os, sys, time
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, gold, privacy

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
print(cfg["code"], env, "gold")

In [0]:
start = time.time()
built = {}
t0 = time.time()
built["dim_date"] = gold.build_dim_date(spark, cfg)
print(f"{'dim_date':18} {built['dim_date']:>7} rows  {time.time() - t0:.0f}s")

t0 = time.time()
for name, rows in gold.build_dimensions(spark, cfg).items():
    built[name] = rows
    print(f"{name:18} {rows:>7} rows")
print(f"{'dimensions':18} {time.time() - t0:.0f}s")

t0 = time.time()
for name, rows in gold.build_facts(spark, cfg).items():
    built[name] = rows
    print(f"{name:18} {rows:>7} rows")
print(f"{'facts':18} {time.time() - t0:.0f}s")

masks = privacy.apply(spark, cfg, env)            # gold tables get their masks before anyone reads them
print("masks", masks)

print("TOTAL", built, f"in {time.time() - start:.0f}s")
common.log_run(spark, cfg, "gold", "success", sum(built.values()), str(built))